### basic workflow 

In [38]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GEMINI_API_KEY")


#langsmith tracking
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACKING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT_NAME"] = os.getenv("LANGCHAIN_PROJECT_NAME")



In [39]:
#data ingetsion - from the website we need to scarpe the data 

from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://docs.langchain.com/langsmith/administration-overview")

loader

In [40]:
docs=loader.load()

In [41]:
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview', 'title': 'Overview - Docs by LangChain', 'language': 'en'}, page_content='Overview - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationAccount administrationOverviewGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusOn this pageResource hierarchyOrganizationsWorkspacesApplicationsResourcesAdditional infoResource tagsUser management and RBACUsersAPI keysExpiration datesPer

In [42]:
##load data --> docs --> divide into chunks -->tetx-->vectors--> create vector store --> retriever --> llm chain

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents =text_splitter.split_documents(docs)

In [43]:
documents

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview', 'title': 'Overview - Docs by LangChain', 'language': 'en'}, page_content='Overview - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationAccount administrationOverviewGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusOn this pageResource hierarchyOrganizationsWorkspacesApplicationsResourcesAdditional infoResource tagsUser management and RBACUsersAPI keysExpiration datesPer

In [68]:
# from langchain_google_genai import GoogleGenerativeAIEmbeddings

# embeddings = GoogleGenerativeAIEmbeddings(
#     model="embedding-001"   # ✅ REQUIRED
# )

In [64]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

C:\Users\prakr\AppData\Local\Temp\ipykernel_9960\4246038141.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4434.86it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [66]:
from langchain_community.vectorstores import FAISS

vectorstoredb = FAISS.from_documents(documents, embeddings)

In [67]:
vectorstoredb

In [69]:
query = "Roles are used to define the set of permissions that a user has within a workspace."
result = vectorstoredb.similarity_search(query)
result

[Document(id='06b621ae-982f-48f9-b5e2-b65e0ba00e01', metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview', 'title': 'Overview - Docs by LangChain', 'language': 'en'}, page_content='Roles are used to define the set of permissions that a user has within a workspace. There are three built-in system roles that cannot be edited:'),
 Document(id='4897770e-de26-4851-aae6-ded01e5596b9', metadata={'source': 'https://docs.langchain.com/langsmith/administration-overview', 'title': 'Overview - Docs by LangChain', 'language': 'en'}, page_content='Workspace Admin has full access to all resources within the workspace.\nWorkspace Editor has full permissions except for workspace management (adding/removing users, changing roles, configuring service keys).\nWorkspace Viewer has read-only access to all resources within the workspace.\n\nOrganization admins can also create/edit custom roles with specific permissions for different resources.\nYou can manage roles under Organiz

In [70]:
result[0].page_content

'Roles are used to define the set of permissions that a user has within a workspace. There are three built-in system roles that cannot be edited:'

In [77]:
from langchain_google_genai import GoogleGenerativeAI
llm = GoogleGenerativeAI(model="gemini-1.5-pro")


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [94]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
# from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)
# Force v1 to avoid the 404 beta error
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", version="v1")


# 2. Define the prompt with the required {context} placeholder
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer the question based ONLY on the context provided below:\n\n{context}"),
    ("human", "{input}")
])

# 3. Create the chain
chain = create_stuff_documents_chain(llm, prompt)

# 4. Your documents
docs = [
    Document(page_content="Workspace Admin has full access to all resources."),
    Document(page_content="Workspace Viewer has read-only access.")
]

# 5. Run the chain
# We pass 'input' for the user question and 'context' for the docs
result = chain.invoke({
    "input": "What is the difference between an Admin and a Viewer?",
    "context": docs
})

print(result)

Unexpected argument 'version' provided to ChatGoogleGenerativeAI. Did you mean: 'output_version'?
c:\Users\prakr\OneDrive\local_dekstop\langchain\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3747: UserWarning: WARNING! version is not default parameter.
                version was transferred to model_kwargs.
                Please confirm that version is what you intended.
  exec(code_obj, self.user_global_ns, self.user_ns)
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


A Workspace Admin has full access to all resources, while a Workspace Viewer has read-only access.
